In [1]:
import torch

from rlaopt.atoms import Box, LinearRegression
from rlaopt.data import DataLoader, Dataset
from rlaopt.expression import Variable
from rlaopt.linalg import NystromConfig
from rlaopt.solvers import ProxGrad, ProxGradConfig, Sapphire, SapphireConfig

In [ ]:
torch.manual_seed(0)
# torch.set_default_dtype(torch.float64)
torch.set_default_device("cuda:0")

In [ ]:
n, p = 100024, 256
s = 10

In [ ]:
X = torch.randn(n, p)
wStar = torch.randn(p) / (p) ** 0.5
# wStar = torch.zeros(p)
# J = torch.randperm(p)[0:s]
# wStar[J] = torch.randn(s) / (s) ** (0.5)
y = X @ wStar + 0.01 * torch.randn(n)

In [ ]:
L = 2 / n * torch.linalg.norm(X, ord=2) ** 2

In [ ]:
L

In [ ]:
dataset = Dataset(X, y, device="cuda:0")
loader = DataLoader(
    dataset, batch_size=1024, shuffle=True, generator=torch.Generator(device="cuda:0")
)

In [ ]:
beta = Variable(
    torch.zeros(
        p,
    ),
    name="beta",
)

In [ ]:
model = LinearRegression(beta, loader, fit_intercept=False)

In [ ]:
scaling = 0.1 / n * torch.linalg.norm(X.T @ y, ord=torch.inf)

In [ ]:
obj = model + Box(beta, -2.0, 1.0)

In [ ]:
config = ProxGradConfig(eta0=0.1, use_linesearch=False)
opt = ProxGrad(obj, config)

In [ ]:
var_vals = model.variable_values
state = opt.init_state(var_vals)

In [ ]:
for _ in range(200):
    var_vals, state = opt.step(var_vals, state)

In [ ]:
state.err

In [ ]:
obj.evaluate(var_vals)

In [ ]:
opt._op_split.f._loss_fn.reduction

In [ ]:
var_vals["beta"]

In [ ]:
state.err

In [ ]:
state.eta

In [ ]:
results = opt.solve(model.variable_values)

In [ ]:
prec_config = NystromConfig(
    rank_init=10, rank_max=10, base_damping=1e-3, error_tolerance=100
)
config = SapphireConfig(
    eta0=0.1, base_method="svrg", grad_batch_size=1024, subproblem_iters=5
)

In [ ]:
opt = Sapphire(obj, config)

In [ ]:
results = opt.solve(model.variable_values)

In [ ]:
obj.evaluate(results.variable_values)

In [ ]:
results.variable_values["beta"]

In [ ]:
results.convergence_status.value